# Latent Transformer Model for Frame Prediction

Latent transformer experiment for Elastic Disks frame forecasting. The VAE compresses frames into vectors, and the transformer predicts the next latent from recent history.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/jordanshivers/generative-video-forecasting.git"
REPO_DIRNAME = "generative-video-forecasting"
INSTALL_REQUIREMENTS = True
RETRAIN = False

_current = Path.cwd().resolve()
REPO_ROOT = None
for _candidate in [_current, *_current.parents]:
    if (_candidate / "requirements.txt").exists() and (_candidate / "src" / "video_forecasting").is_dir():
        REPO_ROOT = _candidate
        break

if REPO_ROOT is None:
    _in_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
    if _in_colab:
        _target = Path("/content") / REPO_DIRNAME
        if not _target.exists():
            subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(_target)])
        for _candidate in [_target, *_target.parents]:
            if (_candidate / "requirements.txt").exists() and (_candidate / "src" / "video_forecasting").is_dir():
                REPO_ROOT = _candidate
                break
    if REPO_ROOT is None:
        raise FileNotFoundError(
            f"Could not find repo root from {_current}. Run locally from the repo root/notebooks "
            "or set GITHUB_REPO_URL for standalone Colab execution."
        )

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])

from video_forecasting.runtime import get_data_dir, get_output_dir

DATA_DIR = get_data_dir(REPO_ROOT)
OUTPUT_DIR = get_output_dir("train_elastic_disks_latent_transformer", REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
import imageio.v2 as imageio
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader

from video_forecasting.runtime import get_device, set_seed
from video_forecasting.presets import batch_size_for_device, get_preset

set_seed(42)
device = get_device(prefer_mps=True)
print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device}")


In [ ]:
from video_forecasting.datasets.elastic_disks import ElasticDisksDataset
from video_forecasting.models.mdn_rnn import SequenceDataset
from video_forecasting.models.transformer import LatentTransformerForecaster
from video_forecasting.training import (
    FrameOnlyDataset,
    count_parameters,
    evaluate_transformer,
    evaluate_vae,
    train_transformer_epoch,
    train_vae_epoch,
)
from video_forecasting.models.vae import VectorVAE as VAE
from video_forecasting.visualization import (
    display_video,
    generate_transformer_rollout_movie,
    plot_training_curves,
    save_reconstruction_frame,
    set_output_dir,
    visualize_transformer_predictions,
    visualize_vae_reconstructions,
    write_comparison_mp4,
)

set_output_dir(OUTPUT_DIR)


## Load Elastic Disks Data


In [ ]:
print("Loading Elastic Disks datasets...")
dataset_cfg = get_preset("elastic_disks")
num_sequences = dataset_cfg["num_sequences"]
max_sequences = dataset_cfg["max_sequences"]
sequence_length = dataset_cfg["sequence_length"]
frame_separation = dataset_cfg["frame_separation"]

train_dataset = ElasticDisksDataset(
    root=str(DATA_DIR),
    train=True,
    num_sequences=num_sequences,
    sequence_length=sequence_length,
    image_size=dataset_cfg["image_size"],
    num_particles=dataset_cfg["num_particles"],
    boundary="reflecting",
    render_mode=dataset_cfg["render_mode"],
    normalize=True,
    frame_separation=frame_separation,
    seed=42,
    max_sequences=max_sequences,
)

test_dataset = ElasticDisksDataset(
    root=str(DATA_DIR),
    train=False,
    num_sequences=num_sequences,
    sequence_length=sequence_length,
    image_size=dataset_cfg["image_size"],
    num_particles=dataset_cfg["num_particles"],
    boundary="reflecting",
    render_mode=dataset_cfg["render_mode"],
    normalize=True,
    frame_separation=frame_separation,
    seed=42,
    max_sequences=max_sequences,
)

print("Elastic Disks split:")
print(f"  Train: {len(train_dataset.sequences)} sequences")
print(f"  Test: {len(test_dataset.sequences)} sequences")
print(f"Train samples: {len(train_dataset)}; test samples: {len(test_dataset)}")


## Initialize VAE and Transformer Models

In [ ]:
vae_cfg = get_preset("vae_vector")
transformer_cfg = get_preset("latent_transformer")
latent_dim = transformer_cfg["latent_dim"]
vae_hidden_dims = vae_cfg["hidden_dims"]
max_initial_spatial_size = 16
vae_beta = 1.0
vae_learning_rate = vae_cfg["learning_rate"]
vae_num_epochs = vae_cfg["num_epochs"]

context_size = transformer_cfg["context_size"]
transformer_model = LatentTransformerForecaster(
    latent_dim=latent_dim,
    d_model=transformer_cfg["d_model"],
    nhead=transformer_cfg["nhead"],
    num_layers=transformer_cfg["num_layers"],
    dim_feedforward=transformer_cfg["dim_feedforward"],
    dropout=transformer_cfg["dropout"],
    max_seq_len=sequence_length,
).to(device)

vae = VAE(
    in_channels=1,
    latent_dim=latent_dim,
    hidden_dims=vae_hidden_dims,
    max_initial_spatial_size=max_initial_spatial_size,
).to(device)

print(f"VAE parameters: {count_parameters(vae):,}")
print(f"Transformer parameters: {count_parameters(transformer_model):,}")


## Train VAE Model

In [ ]:
max_height = max(train_dataset.target_height, test_dataset.target_height)
max_width = max(train_dataset.target_width, test_dataset.target_width)
vae_train_dataset = ConcatDataset([
    FrameOnlyDataset(train_dataset, target_height=max_height, target_width=max_width),
    FrameOnlyDataset(test_dataset, target_height=max_height, target_width=max_width),
])

batch_size = batch_size_for_device(device, vae_cfg["batch_size"])
vae_train_loader = DataLoader(
    vae_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"VAE training frames: {len(vae_train_dataset)}")


In [ ]:
optimizer_vae = torch.optim.AdamW(vae.parameters(), lr=vae_learning_rate, weight_decay=1e-4)
scheduler_lr_vae = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_vae,
    max_lr=vae_learning_rate,
    epochs=vae_num_epochs,
    steps_per_epoch=len(vae_train_loader),
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100.0,
)

vae_train_losses = []
vae_train_recon_losses = []
vae_train_kl_losses = []
vae_path = OUTPUT_DIR / "vae_model_latent_transformer.pt"
reconstruction_frames_dir = OUTPUT_DIR / "vae_reconstruction_frames_transformer"
output_mp4_dir = OUTPUT_DIR / "output_mp4s"
reconstruction_frames_dir.mkdir(exist_ok=True)
output_mp4_dir.mkdir(exist_ok=True)
mp4_path = output_mp4_dir / "vae_training_reconstructions_transformer.mp4"
sample_indices = torch.randperm(len(vae_train_dataset))[:1].tolist()

if vae_path.exists() and not RETRAIN:
    print(f"Loading VAE from {vae_path}...")
    vae.load_state_dict(torch.load(vae_path, map_location=device))
    vae.eval()
else:
    print("Training VAE...")
    for epoch in range(vae_num_epochs):
        recon_loss, kl_loss, loss = train_vae_epoch(
            vae, vae_train_loader, optimizer_vae, device, beta=vae_beta
        )
        scheduler_lr_vae.step()
        vae_train_recon_losses.append(recon_loss)
        vae_train_kl_losses.append(kl_loss)
        vae_train_losses.append(loss)
        save_reconstruction_frame(
            vae, vae_train_dataset, sample_indices, epoch, reconstruction_frames_dir, device
        )
        print(f"Epoch {epoch + 1}/{vae_num_epochs}: loss={loss:.4f}, recon={recon_loss:.4f}, kl={kl_loss:.4f}")
    torch.save(vae.state_dict(), vae_path)
    plot_training_curves(
        vae_train_losses,
        output_path=OUTPUT_DIR / "vae_training_curves_transformer.png",
        title="VAE Training Curves - Latent Transformer",
    )
    frame_files = sorted(reconstruction_frames_dir.glob("epoch_*.png"))
    if frame_files:
        frames = [imageio.imread(frame_file) for frame_file in frame_files]
        write_comparison_mp4(frames, mp4_path, fps=2)


if mp4_path.exists():
    display_video(mp4_path)


## Train Latent Transformer

In [ ]:
transformer_batch_size = batch_size_for_device(device, transformer_cfg["sequence_batch_size"])
train_seq_dataset = SequenceDataset(train_dataset)
test_seq_dataset = SequenceDataset(test_dataset)
transformer_train_loader = DataLoader(
    train_seq_dataset,
    batch_size=transformer_batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
transformer_test_loader = DataLoader(
    test_seq_dataset,
    batch_size=transformer_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"Transformer train batches: {len(transformer_train_loader)}")
print(f"Transformer test batches: {len(transformer_test_loader)}")


In [ ]:
transformer_learning_rate = 1e-4
transformer_num_epochs = transformer_cfg["num_epochs"]
transformer_path = OUTPUT_DIR / "latent_transformer_elastic_disks_model.pt"
optimizer_transformer = torch.optim.AdamW(
    transformer_model.parameters(), lr=transformer_learning_rate, weight_decay=1e-4
)
scheduler_transformer = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_transformer, mode="min", factor=0.5, patience=5
)

vae.load_state_dict(torch.load(vae_path, map_location=device))
vae.eval()
train_losses = []
val_losses = []
best_val_loss = float("inf")

if transformer_path.exists() and not RETRAIN:
    print(f"Loading transformer from {transformer_path}...")
    transformer_model.load_state_dict(torch.load(transformer_path, map_location=device))
    transformer_model.eval()
else:
    print("Training latent transformer...")
    for epoch in range(transformer_num_epochs):
        train_loss = train_transformer_epoch(
            transformer_model, vae, transformer_train_loader, optimizer_transformer, device
        )
        val_loss = evaluate_transformer(transformer_model, vae, transformer_test_loader, device)
        scheduler_transformer.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(transformer_model.state_dict(), transformer_path)
        print(f"Epoch {epoch + 1}/{transformer_num_epochs}: train={train_loss:.6f}, val={val_loss:.6f}")
    plot_training_curves(
        train_losses,
        val_losses,
        OUTPUT_DIR / "latent_transformer_training_curves.png",
        title="Latent Transformer Training Curves",
    )


## Evaluation, Visualizations, and Rollouts

In [ ]:
vae_path = OUTPUT_DIR / "vae_model_latent_transformer.pt"
transformer_path = OUTPUT_DIR / "latent_transformer_elastic_disks_model.pt"

if vae_path.exists() and transformer_path.exists():
    vae.load_state_dict(torch.load(vae_path, map_location=device))
    vae.eval()
    transformer_model.load_state_dict(torch.load(transformer_path, map_location=device))
    transformer_model.eval()

    visualize_vae_reconstructions(
        vae,
        test_dataset,
        num_samples=4,
        device=device,
        title_prefix="Elastic Disks Latent Transformer - ",
    )
    visualize_transformer_predictions(
        transformer_model,
        vae,
        test_seq_dataset,
        num_samples=4,
        num_context_frames=context_size,
        device=device,
        title_prefix="Elastic Disks Latent Transformer - ",
    )
else:
    print("Missing model files; run the training cells first.")
    if not vae_path.exists():
        print(f"  Missing: {vae_path}")
    if not transformer_path.exists():
        print(f"  Missing: {transformer_path}")


In [ ]:
if vae_path.exists() and transformer_path.exists():
    test_sequence_idx = 0
    test_sequence = test_dataset.sequences[test_sequence_idx]
    rollout_path = generate_transformer_rollout_movie(
        transformer_model,
        vae,
        sequence=test_sequence,
        context_size=context_size,
        num_predictions=20,
        start_frame=0,
        device=device,
        fps=10,
        output_dir=str(OUTPUT_DIR / "output_mp4s"),
    )
    print(f"Rollout movie saved to: {rollout_path}")
    display_video(rollout_path)
else:
    print("Missing model files; run the training cells first.")
